# NB07 — Redes Neuronales Profundas (Unidad III)

**Cobertura del syllabus — Unidad III · Fundamentos de Redes Neuronales Artificiales:**
- Modelo de neurona artificial (perceptrón)
- Redes neuronales superficiales (1 capa oculta)
- Redes neuronales profundas (5+ capas con BatchNorm + Dropout)
- Algoritmos de entrenamiento: comparación Adam vs SGD vs RMSprop
- Aplicaciones: predicción de rendimiento de café

**Datos:** `master_cafe_municipal_mensual.csv` (generado tras cargar BD).

**Comparación:** MLP profundo vs XGBoost (NB02 baseline 2da entrega).

**Métricas esperadas:** R² > 0.5 (vs 0.067 baseline) — mejora viene de más datos municipales.

In [ ]:
# ─────────────── 0. Setup ───────────────
import os, sys, warnings, json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers, regularizers
tf.get_logger().setLevel('ERROR')
warnings.filterwarnings('ignore')

RNG = 42
np.random.seed(RNG); tf.random.set_seed(RNG)

PROJECT = Path('..').resolve()
DIR_DATOS = PROJECT / '01_datos' / 'procesados'
DIR_MODELS = PROJECT / '04_modelos_entrenados'; DIR_MODELS.mkdir(parents=True, exist_ok=True)
DIR_FIG = PROJECT / '05_resultados' / 'figuras'; DIR_FIG.mkdir(parents=True, exist_ok=True)
DIR_TABLAS = PROJECT / '05_resultados' / 'tablas'; DIR_TABLAS.mkdir(parents=True, exist_ok=True)

print('TF version:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

## 1. Carga de datos · 2 alternativas

1. **Vía CSV** (si todavía no se cargó la BD): `master_cafe_municipal_mensual.csv`
2. **Vía PostgreSQL** (recomendado): query a `cafe.vw_master_municipal_mensual`

In [ ]:
# Opción A: CSV
csv_master = DIR_DATOS / 'master_cafe_municipal_mensual.csv'
csv_2da    = PROJECT.parent / 'IA_Segunda_Entrega' / 'datasets' / 'master_cafe_semestral.csv'

if csv_master.exists():
    df = pd.read_csv(csv_master)
    print('✓ master municipal:', df.shape)
elif csv_2da.exists():
    df = pd.read_csv(csv_2da)
    print('⚠ usando master semestral (2da entrega):', df.shape)
else:
    raise FileNotFoundError('Sin master_cafe_*.csv — corre primero NB04 o BD')

# Opción B (alternativa): vía Postgres
# import psycopg2
# conn = psycopg2.connect(host='localhost', user='postgres', password='root', dbname='cafe_ia')
# df = pd.read_sql('SELECT * FROM cafe.vw_master_municipal_mensual', conn)

print(df.dtypes.value_counts())
df.head()

In [ ]:
# ─────────────── 2. Preparación de features ───────────────
# Target: rendimiento_ton_ha
TARGET = 'rendimiento_ton_ha'

# Columnas a excluir (identificadores y target)
EXCLUIR = {TARGET, 'cod_mun', 'municipio', 'departamento', 'fecha', 'estado_fisico'}

# Detectar columnas numéricas
num_cols = [c for c in df.select_dtypes(include='number').columns if c not in EXCLUIR]
cat_cols = [c for c in df.select_dtypes(include=['object','category']).columns if c not in EXCLUIR]

print(f'Numéricas: {len(num_cols)} | Categóricas: {len(cat_cols)}')
print('Numéricas:', num_cols)

# Limpiar nulos en target
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
print(f'Filas con target válido: {len(df)}')

# Encoding categóricas
for c in cat_cols:
    df[c + '_enc'] = LabelEncoder().fit_transform(df[c].astype(str))
    num_cols.append(c + '_enc')

X = df[num_cols].copy().fillna(df[num_cols].median(numeric_only=True))
y = df[TARGET].copy()

print('Shape final:', X.shape, y.shape)

In [ ]:
# Split temporal (no aleatorio — para evitar leakage)
if 'anio' in df.columns:
    train_mask = df['anio'] <= 2022
    test_mask  = df['anio'] >  2022
    X_train, X_test = X[train_mask], X[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RNG)

scaler_X = StandardScaler().fit(X_train)
X_train_s = scaler_X.transform(X_train)
X_test_s  = scaler_X.transform(X_test)

print(f'Train: {X_train_s.shape}  Test: {X_test_s.shape}')

## 3. Modelo 1: Perceptrón simple (neurona artificial)
Implementación didáctica del modelo más básico — Unidad III.

In [ ]:
def perceptron_simple(input_dim: int) -> tf.keras.Model:
    """Una sola neurona — el modelo más básico de la Unidad III."""
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(1, activation='linear', name='neurona_unica')(inp)
    return models.Model(inp, x, name='perceptron')

modelo_perc = perceptron_simple(X_train_s.shape[1])
modelo_perc.compile(optimizer='adam', loss='mse', metrics=['mae'])
modelo_perc.summary()

## 4. Modelo 2: Red superficial (1 capa oculta)

In [ ]:
def red_superficial(input_dim: int, hidden: int = 32) -> tf.keras.Model:
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(hidden, activation='relu')(inp)
    x = layers.Dense(1, activation='linear')(x)
    return models.Model(inp, x, name='red_superficial')

modelo_sup = red_superficial(X_train_s.shape[1])
modelo_sup.compile(optimizer='adam', loss='mse', metrics=['mae'])
modelo_sup.summary()

## 5. Modelo 3: Red profunda (5 capas + BatchNorm + Dropout + L2)
Implementa todas las técnicas de regularización del syllabus.

In [ ]:
def red_profunda(input_dim: int) -> tf.keras.Model:
    """
    MLP profundo con técnicas modernas:
      - 5 capas ocultas (256→128→64→32→16)
      - BatchNorm (estabiliza entrenamiento)
      - Dropout 0.3 (regularización por aleatoriedad)
      - L2 (regularización por penalización)
      - LeakyReLU (activación moderna que evita dying ReLU)
    """
    inp = layers.Input(shape=(input_dim,))
    x = inp
    for units in (256, 128, 64, 32, 16):
        x = layers.Dense(units, kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x)
        x = layers.LeakyReLU(0.1)(x)
        x = layers.Dropout(0.3)(x)
    x = layers.Dense(1, activation='linear')(x)
    return models.Model(inp, x, name='mlp_profundo')

modelo_prof = red_profunda(X_train_s.shape[1])
modelo_prof.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='huber',  # robusto a outliers
    metrics=['mae']
)
modelo_prof.summary()

## 6. Entrenamiento — comparación de optimizadores
Demuestra Unidad III: algoritmos de entrenamiento (Adam, SGD, RMSprop).

In [ ]:
OPTS = {
    'Adam':    optimizers.Adam(learning_rate=1e-3),
    'SGD+mom': optimizers.SGD(learning_rate=1e-2, momentum=0.9, nesterov=True),
    'RMSprop': optimizers.RMSprop(learning_rate=1e-3),
}

historias = {}
for nombre, opt in OPTS.items():
    print(f'\n→ Entrenando con {nombre}')
    m = red_profunda(X_train_s.shape[1])
    m.compile(optimizer=opt, loss='huber', metrics=['mae'])
    hist = m.fit(X_train_s, y_train,
                 validation_split=0.15,
                 epochs=120, batch_size=64, verbose=0,
                 callbacks=[
                     callbacks.EarlyStopping(patience=15, restore_best_weights=True),
                     callbacks.ReduceLROnPlateau(patience=7, factor=0.5),
                 ])
    historias[nombre] = hist.history
    yhat = m.predict(X_test_s, verbose=0).flatten()
    print(f'   R²={r2_score(y_test, yhat):.3f}  MAE={mean_absolute_error(y_test, yhat):.3f}')

# Plot
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
for nombre, h in historias.items():
    axs[0].plot(h['loss'],     label=f'{nombre} train')
    axs[0].plot(h['val_loss'], '--', label=f'{nombre} val')
axs[0].set_title('Pérdida — comparación de optimizadores')
axs[0].legend(fontsize=8); axs[0].set_yscale('log')
for nombre, h in historias.items():
    axs[1].plot(h['mae'],     label=f'{nombre} train')
    axs[1].plot(h['val_mae'], '--', label=f'{nombre} val')
axs[1].set_title('MAE — comparación de optimizadores')
axs[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(DIR_FIG / 'NB07_optimizadores.png', dpi=120)
plt.show()

## 7. Comparación final: Perceptrón vs Superficial vs Profundo vs XGBoost

Tabla con métricas — la del MLP profundo debería superar a XGBoost si los datos son suficientes.

In [ ]:
# Entrenar y evaluar los 3 modelos NN
callbacks_default = [
    callbacks.EarlyStopping(patience=15, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(patience=7, factor=0.5),
]

resultados = []
for nombre, builder in [
    ('Perceptrón simple', lambda: perceptron_simple(X_train_s.shape[1])),
    ('Red superficial',   lambda: red_superficial(X_train_s.shape[1])),
    ('MLP profundo',      lambda: red_profunda(X_train_s.shape[1])),
]:
    m = builder()
    m.compile(optimizer='adam', loss='huber', metrics=['mae'])
    m.fit(X_train_s, y_train, validation_split=0.15,
          epochs=120, batch_size=64, verbose=0,
          callbacks=callbacks_default)
    yhat = m.predict(X_test_s, verbose=0).flatten()
    resultados.append({
        'modelo': nombre,
        'rmse': np.sqrt(mean_squared_error(y_test, yhat)),
        'mae':  mean_absolute_error(y_test, yhat),
        'r2':   r2_score(y_test, yhat),
        'mape': np.mean(np.abs((y_test - yhat) / np.maximum(y_test, 1e-3))) * 100,
    })
    if 'profundo' in nombre.lower():
        m.save(DIR_MODELS / 'mlp_profundo_rendimiento.keras')

# Comparar con XGBoost
from xgboost import XGBRegressor
xgb = XGBRegressor(n_estimators=500, learning_rate=0.05,
                   max_depth=5, random_state=RNG)
xgb.fit(X_train, y_train)
yhat_xgb = xgb.predict(X_test)
resultados.append({
    'modelo': 'XGBoost (baseline)',
    'rmse': np.sqrt(mean_squared_error(y_test, yhat_xgb)),
    'mae':  mean_absolute_error(y_test, yhat_xgb),
    'r2':   r2_score(y_test, yhat_xgb),
    'mape': np.mean(np.abs((y_test - yhat_xgb) / np.maximum(y_test, 1e-3))) * 100,
})

df_res = pd.DataFrame(resultados).sort_values('r2', ascending=False)
df_res.to_csv(DIR_TABLAS / 'NB07_comparacion_modelos.csv', index=False)
df_res

## 8. Conclusiones — Unidad III

**Lo que demuestra este notebook:**
1. **Modelo de neurona artificial:** perceptrón simple ✓
2. **Red superficial:** 1 capa oculta ✓
3. **Red profunda:** 5 capas + BN + Dropout + L2 ✓
4. **Algoritmos de entrenamiento:** comparación Adam vs SGD vs RMSprop ✓
5. **Aplicación:** predicción de rendimiento de café ✓

**Mejora esperada con datos municipales:**
- 2da entrega (departamental): R² = 0.067
- Final (municipal + agronómicas + clima satelital): R² > 0.5 esperado

**Next:** NB08 (CNN), NB09 (ML rendimiento detallado), NB10 (LSTM extendido).